# Module 2.2: Clamshell Lid-Close Operation, Screen Blanking, Thermal Capping & SSH Hardening

## Homelab Expansion (Chapter 2) - Step 2: Operating System & Headless Laptop Tuning

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Configure `systemd-logind` so closing the laptop lid (`HandleLidSwitch=ignore`) keeps the server running 24/7 instead of suspending to RAM.
2. Enable Linux kernel `consoleblank` to automatically turn off the laptop's LCD backlight after 60 seconds, saving ~5W of power and preventing panel burn-in.
3. Understand the **ASUS ROG Zephyrus ErgoLift thermal intake design** and position the laptop lid (`15°–20°` slightly ajar or vertically tented) so keyboard intake vents are not choked.
4. Configure CPU frequency governors (`schedutil`/`powersave`) and turbo-boost limits to prevent AC adapter transient over-current shutoffs on a battery-less laptop.
5. Harden `sshd_config` using your existing `id_ed25519_homelab` SSH key from Chapter 1.

## 2.2.1 Conceptual Background: Turning a Gaming Laptop into a True Headless Appliance

Unlike a Raspberry Pi or rackmount server, a laptop has three built-in hardware behaviors that work *against* 24/7 server operation unless explicitly overridden in Linux:

### 1. The Magnetic Hall-Effect Lid Switch (`systemd-logind`)
Laptops contain a tiny magnet in the top display bezel and a Hall-effect sensor in the palmrest. By default, when you close the lid on Ubuntu Server, `systemd-logind` intercepts the ACPI lid event and triggers `suspend` (putting the CPU to sleep and dropping all SSH/Docker connections). Overriding `HandleLidSwitch=ignore` in `/etc/systemd/logind.conf` disables sleep completely.

### 2. The LCD Backlight & Console Blanking (`consoleblank=60`)
When `HandleLidSwitch=ignore` is active (or when the lid is left slightly ajar for airflow), the Linux TTY text console stays illuminated 24/7 unless instructed to blank via VESA DPMS. Passing `consoleblank=60` to the GRUB kernel command line powers down the display panel and backlight 60 seconds after the last physical keypress, dropping idle wall power draw by **4W to 7W**.

### 3. Crucial ROG Zephyrus Airflow Nuance: Keyboard Deck Intake & ErgoLift
The 2019 ASUS ROG Zephyrus (`GA502` / `GU502` / `GX502`) was engineered with two unique thermal features:
1. **Top-Deck Keyboard Air Intake:** The blower fans draw cool air in through both the bottom vents *and* down through the keyboard deck / perforated grill above the function keys.
2. **ErgoLift Hinge:** Opening the lid angles the rear of the laptop up by several millimeters to increase bottom intake clearance.
> 💡 **Best Practice Physical Placement:** Even after setting `HandleLidSwitch=ignore`, either leave the laptop lid propped open at a **15°–30° angle** (putting a small rubber bumper under the rear feet) or place the laptop in a vertical/elevated cooling stand so cool air flows freely over the keyboard deck while the screen remains completely powered off via `consoleblank=60`!

## 2.2.2 Step-by-Step Laboratory Guide

### Task 2.2.1: Disable Lid-Close Suspend in `systemd-logind`
Connect to `homelab-gpu` via SSH and configure `systemd-logind` to ignore lid-close events under all power states:
```bash
sudo tee -a /etc/systemd/logind.conf > /dev/null << 'EOT'
# Homelab 24/7 Server Mode: Never suspend when laptop lid is closed
HandleLidSwitch=ignore
HandleLidSwitchExternalPower=ignore
HandleLidSwitchDocked=ignore
LidSwitchIgnoreInhibited=no
EOT

# Apply immediately without rebooting
sudo systemctl restart systemd-logind
```

### Task 2.2.2: Enable 60-Second LCD Backlight Off (`consoleblank=60`)
Configure GRUB so the laptop screen turns completely off after 60 seconds of keyboard inactivity:
```bash
# 1. Add consoleblank=60 to GRUB_CMDLINE_LINUX_DEFAULT
sudo sed -i 's/GRUB_CMDLINE_LINUX_DEFAULT="/GRUB_CMDLINE_LINUX_DEFAULT="consoleblank=60 /' /etc/default/grub

# 2. Regenerate the GRUB boot configuration
sudo update-grub

# 3. Also trigger immediate 60-second blanking on the current active TTY1 console
sudo sh -c 'TERM=linux setterm --blank 1 --powerdown 1 </dev/tty1 >/dev/tty1' 2>/dev/null || true
```
Verify that the kernel parameter is queued in `/boot/grub/grub.cfg`:
```bash
grep "consoleblank=60" /boot/grub/grub.cfg
```

### Task 2.2.3: Install Hardware Sensors & Configure Battery-Less Power Safety
Because we removed the internal Li-Po battery in Module 1.1, we want to ensure the CPU does not spike simultaneously with the NVIDIA GPU past the AC brick's instantaneous wattage limit.
```bash
# 1. Update apt cache, upgrade packages, and install thermal/power telemetry tools
sudo apt update && sudo apt full-upgrade -y
sudo apt install -y lm-sensors linux-tools-common linux-tools-generic nvme-cli btop

# 2. Auto-detect motherboard/CPU/NVMe thermal sensors
sudo sensors-detect --auto

# 3. Inspect live CPU, NVMe SSD, and ASUS motherboard temperatures
sensors
```
 verify that idle CPU and NVMe temperatures sit comfortably around **35°C – 48°C** thanks to your fresh thermal paste and `K5 PRO` putty from Module 1.2!

### 🖼️ Task 2.2.3 Verification: Thermal Sensor & Lid-Close Configuration Check

**Verification Objective:** Capture a screenshot of your SSH session on `homelab-gpu` showing the output of `sensors` (confirming healthy CPU and NVMe temperatures after your repaste) and `tail -n 5 /etc/systemd/logind.conf`.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 2.2.4: Copy Your Homelab Ed25519 SSH Key & Harden `sshd`
In Chapter 1, you generated `~/.ssh/id_ed25519_homelab` on your workstation. Let's authorize that exact same cryptographic key on `homelab-gpu` and disable password-based SSH logins:

1. **From your Workstation Terminal** (macOS / Linux / Windows PowerShell), copy your public key to `homelab-gpu`:
   - **macOS / Linux / Git Bash:**
     ```bash
     ssh-copy-id -i ~/.ssh/id_ed25519_homelab.pub pi@homelab-gpu.local
     ```
   - **Windows PowerShell:**
     ```powershell
     type $env:USERPROFILE\.ssh\id_ed25519_homelab.pub | ssh pi@homelab-gpu.local "mkdir -p ~/.ssh && chmod 700 ~/.ssh && cat >> ~/.ssh/authorized_keys && chmod 600 ~/.ssh/authorized_keys"
     ```
2. **Test Key-Based Login from your Workstation:**
   ```bash
   ssh -i ~/.ssh/id_ed25519_homelab pi@homelab-gpu.local
   ```
3. **Once logged into `homelab-gpu` with your key**, lock down SSH via a drop-in configuration file:
   ```bash
   sudo tee /etc/ssh/sshd_config.d/99-homelab-hardening.conf > /dev/null << 'EOT'
   PasswordAuthentication no
   KbdInteractiveAuthentication no
   PermitRootLogin no
   PubkeyAuthentication yes
   EOT

   sudo sshd -t && sudo systemctl reload ssh
   ```

### 🖼️ Task 2.2.4 Verification: Passwordless Ed25519 SSH Login & `btop` Dashboard

**Verification Objective:** Capture a screenshot showing your passwordless SSH login into `homelab-gpu` and the `btop` resource monitor displaying all CPU threads and 16GB/32GB system memory.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] `HandleLidSwitch=ignore` active in `/etc/systemd/logind.conf`.
- [ ] `consoleblank=60` added to `/etc/default/grub` so the laptop LCD turns off automatically after 60 seconds.
- [ ] `lm-sensors` installed and verified showing healthy post-repaste idle temperatures.
- [ ] Workstation `id_ed25519_homelab` SSH key authorized on `homelab-gpu` and password SSH authentication disabled.
- [ ] Verification screenshots captured and attached above.

Proceed to [Module 3.1: UniFi Switch Enterprise 8 PoE Topology, Static IP Plan & Unified Mesh](../03_network_expansion_and_cluster_topology/03_1_switch_topology_and_subnet_design.ipynb).